In [25]:
# Import the necessary libraries
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF
from scipy.stats import norm
import pandas as pd

from enum import Enum

In [63]:
class DataSet(Enum):
    ONE = 1
    TWO = 2
    THREE = 3
    FOUR = 4
    FIVE = 5
    SIX = 6
    SEVEN = 7
    EIGHT = 8

data_set = DataSet.ONE;

class AquisitionFunction(Enum):
    UCB = 1


In [17]:

#Load initial data
match data_set:
    case DataSet.ONE:
        X = np.load("initial_data/function_1/initial_inputs.npy")
        y = np.transpose(np.load("initial_data/function_1/initial_outputs.npy"))

        X = np.append(X, [[0.811933, 0.784473]], 0)
        y = np.append(y, 5.357376e-41)
        
        print(X.shape)
        print(y.shape)
    case DataSet.TWO:
        X = np.load("initial_data/function_2/initial_inputs.npy")
        y = np.transpose(np.load("initial_data/function_2/initial_outputs.npy"))

        X = np.append(X, [[0.818082, 0.926635]], 0)
        y = np.append(y, -0.02537857)
        
        print(X.shape)
        print(y.shape)
    case DataSet.THREE:
        X = np.load("initial_data/function_3/initial_inputs.npy")
        y = np.transpose(np.load("initial_data/function_3/initial_outputs.npy"))

        X = np.append(X, [[0.486552, 0.999089, 0.997841]], 0)
        y = np.append(y, -0.48078736)
        
        print(X.shape)
        print(y.shape)
    case DataSet.FOUR:
        X = np.load("initial_data/function_4/initial_inputs.npy")
        y = np.transpose(np.load("initial_data/function_4/initial_outputs.npy"))

        X = np.append(X, [[0.004832, 0.382262, 0.921803, 0.990838]], 0)
        y = np.append(y, -32.284832)
        
        print(X.shape)
        print(y.shape)
    case DataSet.FIVE:
        X = np.load("initial_data/function_5/initial_inputs.npy")
        y = np.transpose(np.load("initial_data/function_5/initial_outputs.npy"))


        X = np.append(X, [[0.212855, 0.844288, 0.880739, 0.871421]], 0)
        y = np.append(y, 1052.113900)
        
        print(X.shape)
        print(y.shape)
    case DataSet.SIX:
        X = np.load("initial_data/function_6/initial_inputs.npy")
        y = np.transpose(np.load("initial_data/function_6/initial_outputs.npy"))


        X = np.append(X, [[0.023802, 0.008308, 0.294059, 0.011586, 0.015609]], 0)
        y = np.append(y, -2.014607)
        
        print(X.shape)
        print(y.shape)
    case DataSet.SEVEN:
        X = np.load("initial_data/function_7/initial_inputs.npy")
        y = np.transpose(np.load("initial_data/function_7/initial_outputs.npy"))

        X = np.append(X, [[0.104168, 0.456114, 0.256427, 0.215101, 0.372639, 0.722473]], 0)
        y = np.append(y, 1.796145391)
        
        print(X.shape)
        print(y.shape)
    case DataSet.EIGHT:
        X = np.load("initial_data/function_8/initial_inputs.npy")
        y = np.transpose(np.load("initial_data/function_8/initial_outputs.npy"))

        X = np.append(X, [[0.345531, 0.274921, 0.707992, 0.654609, 0.742221, 0.787590, 0.240555, 0.887229]], 0)
        y = np.append(y, 8.5110526203)
        
        print(X.shape)
        print(y.shape)
    case _:
        print('No data set found')

(11, 2)
(11,)


In [65]:
# Week 1 General Code
def MakePrediction(X, y, noise_assumption,rbf_lengthscale, beta, n_samples, AF):
    # Define Model
    kernel = RBF(length_scale=rbf_lengthscale, length_scale_bounds='fixed')
    model = GaussianProcessRegressor(kernel = kernel, alpha=noise_assumption)
    
    # Fit Model
    model.fit(X, y)

    n_features = X.shape[1]

    # Create Sample points for aquisition funciton
    x_s = np.random.rand(n_samples, n_features);
    
    post_mean, post_std = model.predict(x_s, return_std=True)
    post_mean, post_std = post_mean.squeeze(), post_std.squeeze()

    if AF == AquisitionFunction.UCB:
        aquisition_function = post_mean + beta * post_std

    print("Aquisition function Max:", aquisition_function.max())
    print("Aquisition function Min:", aquisition_function.min())
    
    index = aquisition_function.argmax();

    return x_s[index,:]


In [61]:
def display_basic_stats(X, y, x_next):

    # Calc general stat of dataset
    
    mean_y = y.mean();
    std_y = y.std();

    max_y = y.max();
    min_y = y.min();

    max_pVal = (max_y - mean_y)/std_y;
    min_pVal = (mean_y-min_y)/std_y;

    print("Mean and Std of Y: ", mean_y, ", ", std_y)
    print("Max and min of Y: ", max_y, ", ", min_y)
    print("Max and Min P-Val: ", max_pVal, ", ", min_pVal)


    # Calc stas of each point
    
    order = np.flip(np.argsort(y,0)).flatten();
    
    max_y_ind = order[0];
    min_y_ind = order[-1];

    nf = X.shape[0];

    stats_raw = [];
    
    for i in range(nf):
        Index = order[i];
        Y = y[Index]
        Y_Rank = i+1;
        Max_ED = np.linalg.norm(X[Index,:]-X[max_y_ind,:])
        Max_YD = y[max_y_ind]-y[Index]
        Min_ED = np.linalg.norm(X[Index,:]-X[min_y_ind,:])
        Min_YD = y[min_y_ind]-y[Index]
        pVal = (y[Index]-mean_y)/std_y
        stats_raw.append([Index, Y, Y_Rank, Max_ED, Max_YD, Min_ED, Min_YD, pVal]);
    
    headings = ['Index', 'Y', 'Y-Rank', 'Max-Euclid-Dist', 'Max-Y-Delta', 'Min-Euclid-Dist', 'Min-Y-Delta', 'P-Val']
    stats = pd.DataFrame(stats_raw, columns = headings)
    print(stats)

    # Calc Stats of new point
    
    dist = np.linalg.norm(X - x_next, axis=1);
    order_n = np.argsort(dist,0)
    
    stats_new = []
    
    for i in range(nf):
        Index = order_n[i];
        Y = y[Index];
        Y_rank = np.where(order==Index)[0]
        Dist = dist[Index]
        stats_new.append([Index, Y, Y_rank, Dist])

    headings = ['Index', 'Y', 'Y-Rank', 'Dist']
    stats = pd.DataFrame(stats_new, columns = headings)
    print(stats)


In [71]:
#Cell for processing dataset 1

noise_assumption = 1e-10;
rbf_lengthscale = 0.1;
beta = 1;
n_samples = 1000000;

AF = AquisitionFunction.UCB

# Pre Processing if required

if data_set == DataSet.ONE:
    inliers = y > -0.0001;
    X = X[inliers, :];
    y = y[inliers];
    beta = 0.5


x_next = MakePrediction(X,y,noise_assumption,rbf_lengthscale, beta, n_samples, AF)

display_basic_stats(X, y, x_next)

nF = X.shape[1];

for i in range(nF):
    nD = "{:.6f}".format(x_next[i])
    print("Element: ", i, ", Value=", nD)

    

Aquisition function Max: 0.4999999648142924
Aquisition function Min: 0.0005437489728101013
Mean and Std of Y:  7.710875114502849e-17 ,  2.3132625343508553e-16
Max and min of Y:  7.710875114502849e-16 ,  -2.1592490357331095e-54
Max and Min P-Val:  2.9999999999999996 ,  0.3333333333333332
   Index              Y  Y-Rank  Max-Euclid-Dist   Max-Y-Delta  \
0      2   7.710875e-16       1         0.000000  0.000000e+00   
1      6   2.535001e-40       2         0.136620  7.710875e-16   
2      9   5.357376e-41       3         0.095895  7.710875e-16   
3      1   1.033078e-46       4         0.214784  7.710875e-16   
4      8   6.229856e-48       5         0.214691  7.710875e-16   
5      0   1.322677e-79       6         0.412709  7.710875e-16   
6      7   3.606771e-81       7         0.727428  7.710875e-16   
7      3  3.341771e-124       8         0.480862  7.710875e-16   
8      5  -2.089093e-91       9         0.776583  7.710875e-16   
9      4  -2.159249e-54      10         0.667475  7.